# Konkani Companion — Colab backend runner

Runs the backend in `backend/app` on a Colab GPU and exposes it through ngrok.
No keys live in this notebook.

**One-time setup:** open the 🔑 *Secrets* panel on the left and add:

| Secret | Required | Notes |
|---|---|---|
| `GROQ_API_KEY` | yes | Reasoning model |
| `BACKEND_API_KEY` | yes | Shared secret; put the same value in the frontend's `.env.local` |
| `NGROK_AUTHTOKEN` | yes | ngrok tunnel |
| `NGROK_DOMAIN` | no | Your reserved ngrok domain, so the URL never changes |
| `FIRECRAWL_API_KEY` | no | Primary web search |
| `TAVILY_API_KEY` | no | Fallback web search (DuckDuckGo is the last resort, no key needed) |
| `GITHUB_TOKEN` | only if the repo is private | Read-only token for cloning |

Enable notebook access for each secret. Runtime → Change runtime type → **T4 GPU**.

Run cells 1–4 once per runtime. Re-run only cell 5 to restart the server (models stay loaded).

In [ ]:
# 1. Get the code
import os, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/boybothere/konkani_companion.git"
BRANCH = "main"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if os.path.isdir("/content/kc/.git"):
    subprocess.run(["git", "-C", "/content/kc", "pull", "--ff-only"], check=True)
else:
    token = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/kc"], check=True)
    subprocess.run(["git", "-C", "/content/kc", "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", "/content/kc", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install system packages (OCR + audio decoding) and Python dependencies
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-mar ffmpeg > /dev/null
!pip install -q unsloth
!pip install -q -r /content/kc/backend/requirements.txt -r /content/kc/backend/requirements-gpu.txt pyngrok
!tesseract --list-langs
# If pip asks you to restart the runtime, do it, then continue from cell 3.

In [ ]:
# 3. Load secrets into the environment (values are never printed)
import os
from google.colab import userdata

NAMES = ["GROQ_API_KEY", "BACKEND_API_KEY", "NGROK_AUTHTOKEN", "NGROK_DOMAIN",
         "FIRECRAWL_API_KEY", "TAVILY_API_KEY"]
for name in NAMES:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value and value.strip():
        os.environ[name] = value.strip()  # pasted values often carry a trailing newline
print({name: ("set" if os.environ.get(name) else "missing") for name in NAMES})
for required in ("GROQ_API_KEY", "BACKEND_API_KEY", "NGROK_AUTHTOKEN"):
    assert os.environ.get(required), f"Add the {required} secret first."

# Optional overrides (see backend/.env.example for all settings)
# os.environ["WHISPER_MODEL"] = "large-v3"
# os.environ["SEARCH_PROVIDERS"] = "tavily,duckduckgo"

In [ ]:
# 3b. (Optional) Reload secrets. Run this if you added or changed a secret after
#     running cell 3, or if cell 4 says a key "is not set". Then run cell 4 again.
import os
from google.colab import userdata

for name in ["GROQ_API_KEY", "BACKEND_API_KEY", "NGROK_AUTHTOKEN", "NGROK_DOMAIN",
             "FIRECRAWL_API_KEY", "TAVILY_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name).strip()
        print(name, "loaded")
    except Exception as e:  # not found, or "Notebook access" is off for this secret
        print(name, "FAILED:", type(e).__name__, e)

# Settings are cached once read; forget them so cell 4 sees the new values.
try:
    from app.config import get_settings
    get_settings.cache_clear()
except ImportError:
    pass  # cell 4 hasn't run yet: nothing cached

In [ ]:
# 4. Load the models (Qwen LoRA + Whisper). The first run takes a few minutes.
#    Re-running it (after `git pull` in cell 1) reloads the code but keeps the
#    models already on the GPU, so it is quick and can't run out of GPU memory.
import os, sys, logging
sys.path.insert(0, "/content/kc/backend")
previous = globals().get("services")  # models from an earlier run of this cell
# Forget previously imported backend code, so re-running this cell after
# `git pull` (cell 1) really loads the new version.
for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[name]
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s", force=True)

# Load the secrets here too, so this cell works even in a fresh session where
# cell 3 hasn't run. A missing or unshared secret is reported by name.
from google.colab import userdata
for name in ["GROQ_API_KEY", "BACKEND_API_KEY", "NGROK_AUTHTOKEN", "NGROK_DOMAIN",
             "FIRECRAWL_API_KEY", "TAVILY_API_KEY"]:
    if os.environ.get(name):
        continue
    try:
        value = userdata.get(name)
    except Exception as exc:  # not found, or notebook access not granted
        print(f"{name}: not loaded ({type(exc).__name__})")
        continue
    if value and value.strip():
        os.environ[name] = value.strip()
assert os.environ.get("GROQ_API_KEY"), (
    "GROQ_API_KEY is missing: check its name in the Secrets panel and that notebook access is on."
)

from app.config import get_settings
from app.pipeline import build_services

if previous is not None:
    services = build_services(get_settings(), translator=previous.translator, stt=previous.stt)
    print("Reused the models already loaded; code reloaded.")
else:
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()
    try:
        services = build_services(get_settings())
    except torch.cuda.OutOfMemoryError:
        raise SystemExit(
            "GPU is full, probably from an earlier failed run. "
            "Runtime > Restart session, then run cells 3, 4 and 5 (not 2)."
        ) from None
print("Models loaded. Search providers:", services.search.configured_providers())

In [ ]:
# 5. Start the API and the ngrok tunnel. Interrupt this cell to stop the server.
import os
import uvicorn
from pyngrok import ngrok
from app.main import create_app
from app.tunnel import open_tunnel, set_auth_token  # work with old and new ngrok agents

app = create_app(services)
set_auth_token(os.environ["NGROK_AUTHTOKEN"], ngrok)
ngrok.kill()
public_url = open_tunnel(8000, os.environ.get("NGROK_DOMAIN"), ngrok)
print("PUBLIC URL:", public_url)
print("Set BACKEND_URL to this in the frontend's .env.local")

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
try:
    await server.serve()
finally:
    ngrok.kill()

### Troubleshooting

| You see | Do this |
|---|---|
| `GROQ_API_KEY is not set` (or another key) | Run **3b**, then cell 4 again. If 3b says `FAILED`, check the secret's name and that **Notebook access** is on. |
| `CUDA out of memory` | Runtime → **Restart session**, then run cells 3, 4, 5 (not 2). |
| Updated the code on GitHub | Run cells **1, 4, 5**. Cell 4 reloads the code and keeps the models loaded. |
| `ERR_NGROK_334` (domain in use) | Another session is using the domain. Stop it (or check dashboard.ngrok.com → Endpoints), wait ~30 s, run cell 5 again. |
| Check what's running | Open `https://<your-ngrok-domain>/health`: `version` shows the running commit. |